# ユニットテストを Colab で走らせる

ローカル PC がメモリ枯渇で落ちるため、テストの実行環境を Colab に逃がす。

上から順に実行するだけでよい。`BRANCH` だけ必要に応じて書き換える。

- 認証は不要（リポジトリは公開）
- `chromadb` は入れない。旧 `udemy3.py` だけが import しており、`testpaths = tests` の対象外
- `-m "not integration"` は `pytest.ini` の既定。実機 OCR と Ollama を要するテストは走らない
- したがって GPU は使わないが、L4 ランタイムのままでも支障はない

In [41]:
REPO = "https://github.com/Hide369/local-llm-rag.git"
BRANCH = "chore/drop-stale-chromadb-claims"
WORKDIR = "/content/local-llm-rag"

In [42]:
# 1. 取得（2回目以降は最新を取り直す）
import os, subprocess, sys

def run(*args, cwd=None):
    print("$", " ".join(args))
    r = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    print(r.stdout + r.stderr)
    if r.returncode:
        raise SystemExit(f"failed: {' '.join(args)}")

if not os.path.isdir(WORKDIR):
    run("git", "clone", "--branch", BRANCH, REPO, WORKDIR)
else:
    run("git", "fetch", "origin", BRANCH, cwd=WORKDIR)
    run("git", "checkout", BRANCH, cwd=WORKDIR)
    run("git", "reset", "--hard", f"origin/{BRANCH}", cwd=WORKDIR)

run("git", "log", "--oneline", "-3", cwd=WORKDIR)
print(sys.version)

$ git fetch origin chore/drop-stale-chromadb-claims
From https://github.com/Hide369/local-llm-rag
 * branch            chore/drop-stale-chromadb-claims -> FETCH_HEAD
 * [new branch]      chore/drop-stale-chromadb-claims -> origin/chore/drop-stale-chromadb-claims

$ git checkout chore/drop-stale-chromadb-claims
Branch 'chore/drop-stale-chromadb-claims' set up to track remote branch 'chore/drop-stale-chromadb-claims' from 'origin'.
Switched to a new branch 'chore/drop-stale-chromadb-claims'

$ git reset --hard origin/chore/drop-stale-chromadb-claims
HEAD is now at 6976da3 chore: point the Colab run at this branch

$ git log --oneline -3
6976da3 chore: point the Colab run at this branch
60d9f3c docs: name the current store in comments that still said ChromaDB
7f238f1 refactor: drop the $and wrapper the new store does not need

3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]


In [43]:
# 2. 依存の導入。chromadb は requirements から除去済みだが、古いコミットを
#    検証するときのために念のため除外する（通常は何もしない）
req = os.path.join(WORKDIR, "requirements.txt")
trimmed = "/content/requirements-colab.txt"
with open(req, encoding="utf-8") as f:
    lines = [l for l in f if not l.strip().startswith("chromadb")]
with open(trimmed, "w", encoding="utf-8") as f:
    f.writelines(lines)
print("".join(l for l in lines if l.strip() and not l.startswith("#")))

run(sys.executable, "-m", "pip", "install", "-q", "-r", trimmed)

pymupdf==1.28.2
python-pptx==1.0.2
python-docx==1.2.0
rapidocr==3.9.2
onnxruntime==1.28.0
huggingface_hub==1.27.0
tokenizers==0.23.1
numpy==2.5.2
langchain-text-splitters==1.1.2
streamlit==1.61.1
requests==2.32.3
python-dotenv==1.2.2
pytest==9.1.1
pillow==12.3.0

$ /usr/bin/python3 -m pip install -q -r /content/requirements-colab.txt



In [44]:
# 3. テスト実行。pytest.ini を効かせるためリポジトリ直下で走らせる
r = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--tb=short"],
    cwd=WORKDIR, text=True, capture_output=True,
)
print(r.stdout[-20000:])
print(r.stderr[-4000:])
print("exit code:", r.returncode)
assert r.returncode == 0, "テストが失敗しています。上の出力を確認してください。"

........................................................................ [ 18%]
........................................................................ [ 36%]
........................................................................ [ 55%]
........................................................................ [ 73%]
........................................................................ [ 92%]
...............................                                          [100%]
391 passed, 3 deselected in 35.67s


exit code: 0
